# Outline

- Test the best model from model selection
- Make Pickle file for pipeline and model

In [1]:
import numpy as np
import pandas as pd


In [2]:
X_train = pd.read_csv('../4. Missing Values Imputation/Flats/X_train.csv',index_col=0).drop(columns=['Price per Unit','Elevators','Floor in Building','Floor','Store Rooms','luxury_type','Building Type'])
X_test = pd.read_csv('../4. Missing Values Imputation/Flats/X_test.csv',index_col=0).drop(columns=['Price per Unit','Elevators','Floor in Building','Floor','Store Rooms','luxury_type','Building Type'])
y_train = pd.read_csv('../4. Missing Values Imputation/Flats/y_train.csv',index_col=0)
y_test = pd.read_csv('../4. Missing Values Imputation/Flats/y_test.csv',index_col=0)

In [3]:
X_train.head()

,Bath(s),Area(Marla),Bedroom(s),Servant Quarters,Kitchens,Building,Main Location,Parking Spaces,Property era,Floor Level,Elevator Capacity
6148,1.0,1.0,1.0,1.0,1.0,Gulberg Residencia,Gulberg Residencia,1.0,Recently Built,Ground/First Floor,Single/None
4031,1.0,2.0,1.0,0.0,1.0,D-17,D-17,1.0,Modern Era,Ground/First Floor,Single/None
1222,3.0,4.0,3.0,1.0,1.0,H-13,H-13,1.0,Recently Built,Ground/First Floor,Single/None
1783,2.0,6.0,2.0,1.0,1.0,Top City 1,Top City 1,1.0,Recently Built,Ground/First Floor,Multiple Bank
1550,2.0,3.0,2.0,1.0,1.0,National Police Foundation O-9,National Police Foundation O-9,1.0,Recently Built,Ground/First Floor,Single/None


In [66]:
X_train.columns

Index(['Bath(s)', 'Area(Marla)', 'Bedroom(s)', 'Servant Quarters', 'Kitchens',
       'Building', 'Main Location', 'Parking Spaces', 'Property era',
       'Floor Level', 'Elevator Capacity'],
      dtype='object')

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

In [12]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score,mean_absolute_error
from category_encoders import TargetEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer,RobustScaler

In [6]:
from sklearn import set_config

set_config(transform_output='pandas')

In [7]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer

class CustomOrdinalMapper(BaseEstimator, TransformerMixin):
    def __init__(self, mappings):
        self.mappings = mappings
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        # Handle case if X is a NumPy array, convert to DataFrame for column mapping
        if not isinstance(X, pd.DataFrame):
            X_copy = pd.DataFrame(X)
        else:
            X_copy = X.copy()
            
        for col, mapping_dict in self.mappings.items():
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(mapping_dict).fillna(1).astype(int)
        return X_copy

    def get_feature_names_out(self, input_features=None):
        """Returns the input features as the output features."""
        if input_features is None:
            return np.array(list(self.mappings.keys()))
        return np.asarray(input_features)


class LogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        """Custom Transformer for Log Transformation."""
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = np.copy(X)
        return np.log1p(X_copy)
        
    def inverse_transform(self, X):
        X_copy = np.copy(X)
        return np.expm1(X_copy) 

    def get_feature_names_out(self, input_features=None):
        """Returns the input features as the output features."""
        if input_features is None:
            raise ValueError("input_features must be provided to get_feature_names_out.")
        return np.asarray(input_features)

all_mappings = {
    'Property era': {
        'Vintage': 1, 'Established': 2, 'Recently Built': 3, 
        'Modern Era': 4, 'Brand New (2025)': 5, 'Future': 6
    },
    'Floor Level': {
        'Lower Floors': 1, 'Ground/First Floor': 2, 'Mid-Level': 3, 'High-Rise': 4
    },
    'Elevator Capacity': {
        'Single/None': 1, 'Dual Setup': 2, 'Multiple Bank': 3, 'High-Capacity': 4
    }
}

In [8]:
categorical_cols = ['Property era', 'Floor Level', 'Elevator Capacity']

In [33]:
numeric_transformer = Pipeline(steps=[
    ('Log', FunctionTransformer(np.log1p)),
    ('Scale', RobustScaler())
])

prepocessor = ColumnTransformer(
    [
        #('Custom Ordinal Encoder',CustomOrdinalMapper(all_mappings),categorical_cols),
        ('Target Encoder',TargetEncoder(smoothing=20,min_samples_leaf=2,handle_unknown='value'),['Main Location', 'Building']+categorical_cols),
        ('NumericTransform', numeric_transformer, ['Area(Marla)'])
    ],
    remainder='passthrough',verbose_feature_names_out=False
)

In [34]:
X_train.columns

Index(['Bath(s)', 'Area(Marla)', 'Bedroom(s)', 'Servant Quarters', 'Kitchens',
       'Building', 'Main Location', 'Parking Spaces', 'Property era',
       'Floor Level', 'Elevator Capacity'],
      dtype='object')

In [35]:
prepocessor.fit_transform(X_train,y_train).columns

Index(['Main Location', 'Building', 'Property era', 'Floor Level',
       'Elevator Capacity', 'Area(Marla)', 'Bath(s)', 'Bedroom(s)',
       'Servant Quarters', 'Kitchens', 'Parking Spaces'],
      dtype='object')

In [36]:
y_train_log = np.log1p(y_train)

In [37]:
X_train_transformed = prepocessor.fit_transform(X_train,y_train_log)
X_test_transformed = prepocessor.transform(X_test)

In [26]:
overfitted_params = {'n_estimators': 1000,
 'max_depth': 15,
 'learning_rate': 0.00505825958265549,
 'subsample': 0.8944998368111671,
 'colsample_bytree': 0.9305368372411382,
 'colsample_bylevel': 0.7407153424980754,
 'gamma': 0.021373237297058784,
 'min_child_weight': 4,
 'reg_alpha': 0.005116450915274579,
 'reg_lambda': 0.0007460273123214234,
 'grow_policy': 'lossguide',
 'max_leaves': 52}

improved_params = {
 'n_estimators': 1000,          # Keep this, but make sure to use early_stopping_rounds!
 'max_depth': 5,                # Drastically reduced to stop memorizing rows
 'learning_rate': 0.005,        
 'subsample': 0.89,
 'colsample_bytree': 0.93,
 'colsample_bylevel': 0.74,
 'gamma': 0.1,                  # Increased to prune useless branches
 'min_child_weight': 1,         # Reduced so rare 2-storey units aren't ignored
 'reg_alpha': 0.005,
 'reg_lambda': 0.001,
 'grow_policy': 'lossguide',
 'max_leaves': 15               # Reduced to prevent overly complex trees
}

In [27]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [21]:
best_model = XGBRegressor(**improved_params)
best_model.fit(X_train_transformed,y_train_log['Price(Cr)'],
               eval_set=[(X_train_transformed, y_train_log), (X_test_transformed, y_test_log)],
               verbose=100)
y_pred = best_model.predict(X_test_transformed)
y_pred = np.expm1(y_pred)
r2_score(y_test,y_pred)

[0]	validation_0-rmse:0.60558	validation_1-rmse:0.55410
[100]	validation_0-rmse:0.38965	validation_1-rmse:0.36286
[200]	validation_0-rmse:0.26384	validation_1-rmse:0.25692
[300]	validation_0-rmse:0.19347	validation_1-rmse:0.20322
[400]	validation_0-rmse:0.15548	validation_1-rmse:0.17791
[500]	validation_0-rmse:0.13587	validation_1-rmse:0.16667
[600]	validation_0-rmse:0.12665	validation_1-rmse:0.16223
[700]	validation_0-rmse:0.12249	validation_1-rmse:0.16037
[800]	validation_0-rmse:0.12068	validation_1-rmse:0.15964
[900]	validation_0-rmse:0.11978	validation_1-rmse:0.15937
[999]	validation_0-rmse:0.11914	validation_1-rmse:0.15913


0.9137964248657227

In [22]:
mean_absolute_error(y_test,y_pred)

0.43200695514678955

In [54]:
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

# --- 1. PREPARE THE RAW PKR TARGETS ---
# Multiply by 1 Crore to get raw PKR
y_train_raw = y_train['Price(Cr)'] * 10000000
y_test_raw = y_test['Price(Cr)'] * 10000000

# Apply standard log
y_train_log = np.log(y_train_raw)
y_test_log = np.log(y_test_raw)

# --- 2. RETRAIN THE MODEL FROM SCRATCH ---
# Re-initialize to clear old memory
best_model = XGBRegressor(**improved_params)

best_model.fit(
    X_train_transformed, 
    y_train_log, # Train on the new log-PKR targets
    eval_set=[(X_train_transformed, y_train_log), (X_test_transformed, y_test_log)],
    verbose=100
)

# --- 3. PREDICT AND REVERSE ---
# Predict (outputs will be in log-PKR)
y_pred_log = best_model.predict(X_test_transformed)

# Reverse the log back to raw PKR
y_pred_pkr = np.exp(y_pred_log)

# --- 4. CALCULATE TRUE ERROR ---
r2 = r2_score(y_test_raw, y_pred_pkr)
mae_pkr = mean_absolute_error(y_test_raw, y_pred_pkr)

print("\n--- FINAL RESULTS ---")
print(f"R-Squared Score: {r2:.3f}")
print(f"Average Error: {mae_pkr:,.0f} PKR")

[0]	validation_0-rmse:0.90335	validation_1-rmse:0.84421
[100]	validation_0-rmse:0.58761	validation_1-rmse:0.55841
[200]	validation_0-rmse:0.40491	validation_1-rmse:0.40103
[300]	validation_0-rmse:0.30427	validation_1-rmse:0.32303
[400]	validation_0-rmse:0.25053	validation_1-rmse:0.28635
[500]	validation_0-rmse:0.22251	validation_1-rmse:0.26935
[600]	validation_0-rmse:0.20679	validation_1-rmse:0.26143
[700]	validation_0-rmse:0.19694	validation_1-rmse:0.25729
[800]	validation_0-rmse:0.19007	validation_1-rmse:0.25499
[900]	validation_0-rmse:0.18546	validation_1-rmse:0.25362
[999]	validation_0-rmse:0.18273	validation_1-rmse:0.25296

--- FINAL RESULTS ---
R-Squared Score: 0.916
Average Error: 4,215,123 PKR


In [46]:
test_input = X_train.iloc[:5]

In [47]:
test_input_transformed = prepocessor.transform(test_input)

In [48]:
test_input_transformed

,Main Location,Building,Property era,Floor Level,Elevator Capacity,Area(Marla),Bath(s),Bedroom(s),Servant Quarters,Kitchens,Parking Spaces
6148,0.835680,0.835680,1.023648,1.105514,1.015150,-1.354756,1.0,1.0,1.0,1.0,1.0
4031,0.682126,0.688180,1.207846,1.105514,1.015150,-0.854756,1.0,1.0,0.0,1.0,1.0
1222,0.788549,0.800230,1.023648,1.105514,1.015150,-0.224830,3.0,3.0,1.0,1.0,1.0
1783,0.756197,0.800029,1.023648,1.105514,1.459011,0.190091,2.0,2.0,1.0,1.0,1.0
1550,0.746192,0.746192,1.023648,1.105514,1.015150,-0.500000,2.0,2.0,1.0,1.0,1.0


In [51]:
y_pred_test = best_model.predict(test_input_transformed)
y_pred_test = np.exp(y_pred_test)
r2_score(y_train[:5],y_pred_test)

-591451557724160.0

In [50]:
mean_absolute_error(y_train[:5],y_pred_test)

7939764.0

In [31]:
y_pred_test

array([0.5786303 , 0.4120329 , 0.92286575, 1.4537978 , 0.63125706],
      dtype=float32)

In [88]:
y_train[:5]

,Price(Cr)
6148,0.60
4031,0.37
1222,0.95
1783,1.35
1550,0.47


In [55]:
model_pipeline = Pipeline(
    [
        ('Preprocessor',prepocessor),
        ('Model',best_model)
    ]
)

In [56]:
model_pipeline.fit(X_train,y_train_log)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Preprocessor', ...), ('Model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Target Encoder', ...), ('NumericTransform', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output o

In [57]:
np.exp(model_pipeline.predict(X_train[:5]))

array([ 5283043.5,  3875063.5,  8599278. , 13951020. ,  5622874.5],
      dtype=float32)

In [58]:
# Dump the best pipeline
import joblib

joblib.dump(model_pipeline,'pipeline_pickle_flats_new.joblib')

['pipeline_pickle_flats_new.joblib']

In [59]:
model = joblib.load('pipeline_pickle_flats_new.joblib')

In [62]:
y_pred =model.predict(X_test)
y_pred = np.exp(y_pred)
r2_score(y_test*10000000,y_pred)

0.9152373671531677

In [65]:
mean_absolute_error(y_test*10000000,y_pred)/10000000

0.425277

In [39]:
model = joblib.load('pipeline_pickle_flats.joblib')

In [40]:
np.expm1(model.predict(X_train[X_train['Building']=='The Centaurus'].head(1)))

array([9.020709], dtype=float32)

In [42]:
X_train[X_train['Building']=='The Centaurus'].head(2)

,Bath(s),Area(Marla),Bedroom(s),Servant Quarters,Kitchens,Building,Main Location,Parking Spaces,Property era,Floor Level,Elevator Capacity
5099,3.0,7.0,2.0,1.0,1.0,The Centaurus,F-8,1.0,Recently Built,Ground/First Floor,Multiple Bank
684,3.0,9.0,2.0,1.0,1.0,The Centaurus,F-8,1.0,Recently Built,Ground/First Floor,Multiple Bank


In [43]:
y_train.loc[5099]

Price(Cr)    9.83
Name: 5099, dtype: float64

In [44]:
test_flat = np.array([[4, 10, 2, 1.0, 1.0, 'The Centaurus', 'F-8', 2.0,
        'Recently Built', 'High-Rise',
        'Multiple Bank']])

In [45]:
test_flat = pd.DataFrame(test_flat,columns=X_train.columns)
test_flat = test_flat.astype(X_train.dtypes)

In [46]:
np.expm1(model.predict(test_flat))

array([12.7625065], dtype=float32)

In [62]:
X_train['Bedroom(s)'].max()

5.0